# 第 04 课 · 自行编写二维不可压缩 Navier–Stokes 求解器

> **学习模式：**独立编写 JAX CFD 求解器 | VS Code / Jupyter | 从公式到可运行代码

**本课目标**：基于上节的 MAC 网格与压力投影，完成包含对流、扩散和压力修正的二维求解器，并用 Taylor–Green 涡解析解验证。

**前置知识**：第 03 课的投影算子。

**操作约定**：先逐单元执行，再尝试“实验与练习”；若改变了函数定义或网格设置，请重启内核并执行全部单元格。JAX 第一次调用包含编译时间。

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from jax import lax
jax.config.update("jax_enable_x64", True)
print("JAX:", jax.__version__, "devices:", jax.devices())

## 1. 数值算法

在周期域中采用预测-投影法：

$$
\frac{\mathbf u^* -\mathbf u^n}{\Delta t}
=-(\mathbf u\cdot\nabla)\mathbf u+\nu\nabla_h^2\mathbf u+\mathbf f,
\qquad \nabla_h^2 q=\nabla_h\cdot\mathbf u^*,
\qquad \mathbf u^{n+1}=\mathbf u^*-\nabla_h q.
$$

对流项在交错面上使用邻近面速度插值和中心差分。时间推进采用两阶段显式 RK2，并在每阶段投影。这里优先保证**正确性与可读性**，不是高 Reynolds 数工业级格式。

In [ ]:
def grid(nx=64,ny=32,Lx=8.,Ly=4.):
 dx,dy=Lx/nx,Ly/ny
 x=jnp.arange(nx)*dx; y=jnp.arange(ny)*dy
 X,Y=jnp.meshgrid(x,y,indexing='ij')
 return dx,dy,X,Y

In [ ]:
def div(u,v,dx,dy):
 return (u-jnp.roll(u,1,0))/dx+(v-jnp.roll(v,1,1))/dy

In [ ]:
def grad(p,dx,dy):
 return (jnp.roll(p,-1,0)-p)/dx,(jnp.roll(p,-1,1)-p)/dy

In [ ]:
def lap(q,dx,dy):
 return (jnp.roll(q,-1,0)+jnp.roll(q,1,0)-2*q)/(dx*dx)+(jnp.roll(q,-1,1)+jnp.roll(q,1,1)-2*q)/(dy*dy)

In [ ]:
def symbol(nx,ny,dx,dy):
 kx=2*jnp.pi*jnp.fft.fftfreq(nx,d=dx)[:,None]
 ky=2*jnp.pi*jnp.fft.fftfreq(ny,d=dy)[None,:]
 s=-4*jnp.sin(kx*dx/2)**2/dx**2-4*jnp.sin(ky*dy/2)**2/dy**2
 return jnp.where(s==0,1.,s)

In [ ]:
def solve_poisson(rhs,s):
 rhs_hat=jnp.fft.fftn(rhs)
 p=jnp.fft.ifftn(rhs_hat/s).real
 return p-p.mean()

In [ ]:
def project(u,v,dx,dy,s):
 p=solve_poisson(div(u,v,dx,dy),s)
 px,py=grad(p,dx,dy)
 return u-px,v-py,p

## 2. MAC 面上的对流项

例如 $u$ 方程中：

$$
(\mathbf u\cdot\nabla)u= u\,\partial_x u + v\,\partial_yu.
$$

由于 $u$ 与 $v$ 不存储在同一网格位置，需要对 $v$ 插值到 $u$ 面位置。下方函数中 `v_u` 和 `u_v` 正是这种插值。

In [ ]:
def advection(u,v,dx,dy):
 v_u=0.25*(v+jnp.roll(v,-1,0)+jnp.roll(v,1,1)+jnp.roll(jnp.roll(v,-1,0),1,1))
 u_v=0.25*(u+jnp.roll(u,1,0)+jnp.roll(u,-1,1)+jnp.roll(jnp.roll(u,1,0),-1,1))
 dux=(jnp.roll(u,-1,0)-jnp.roll(u,1,0))/(2*dx)
 duy=(jnp.roll(u,-1,1)-jnp.roll(u,1,1))/(2*dy)
 dvx=(jnp.roll(v,-1,0)-jnp.roll(v,1,0))/(2*dx)
 dvy=(jnp.roll(v,-1,1)-jnp.roll(v,1,1))/(2*dy)
 return u*dux+v_u*duy,u_v*dvx+v*dvy

In [ ]:
def step_navier(u,v,dx,dy,dt,nu,s,masku=None,maskv=None,eta=.02,targetU=0.,body_v=0.,gain=0.):
 def rhs(u,v):
  advu,advv=advection(u,v,dx,dy)
  force=gain*(targetU-jnp.mean(u))
  return -advu+nu*lap(u,dx,dy)+force,-advv+nu*lap(v,dx,dy)
 def filter_and_project(u,v):
  if masku is not None:
   u=u/(1.+dt/eta*masku)
   v=(v+dt/eta*maskv*body_v)/(1.+dt/eta*maskv)
  un,vn,p=project(u,v,dx,dy,s)
  return un,vn,p
 du,dv=rhs(u,v)
 u1,v1,p1=filter_and_project(u+dt*du,v+dt*dv)
 du1,dv1=rhs(u1,v1)
 u2,v2,p=filter_and_project(0.5*(u+u1+dt*du1),0.5*(v+v1+dt*dv1))
 return u2,v2,u1,v1,p

## 3. Taylor–Green 涡的解析解

取 $[0,2\pi]^2$、密度 $\rho=1$：

$$
\begin{aligned}
u(x,y,t)&=\sin x\cos y\,e^{-2\nu t},\\
v(x,y,t)&=-\cos x\sin y\,e^{-2\nu t},\\
p(x,y,t)&=-\frac{1}{4}(\cos2x+\cos2y)e^{-4\nu t}.
\end{aligned}
$$

这是非常适合初学者的不可压缩 Navier–Stokes 验证算例：解析速度随时间衰减，并且始终满足 $\nabla\cdot\mathbf u=0$。

In [ ]:
n=48
L=2*np.pi
dx,dy,_,_=grid(n,n,L,L)
S=symbol(n,n,dx,dy)
nu=0.05;dt=0.01;nsteps=100
xu=(jnp.arange(n)[:,None]+0.5)*dx; yu=jnp.arange(n)[None,:]*dy
xv=jnp.arange(n)[:,None]*dx; yv=(jnp.arange(n)[None,:]+0.5)*dy
u_initial=jnp.sin(xu)*jnp.cos(yu)
v_initial=-jnp.cos(xv)*jnp.sin(yv)
u_initial,v_initial,_=project(u_initial,v_initial,dx,dy,S)
@jax.jit
def simulate_tgv(u,v):
 def advance(i,state):
  a,b=state
  a1,b1,*_=step_navier(a,b,dx,dy,dt,nu,S)
  return (a1,b1)
 return lax.fori_loop(0,nsteps,advance,(u,v))
u_final,v_final=simulate_tgv(u_initial,v_initial)
t=nsteps*dt
u_exact=jnp.sin(xu)*jnp.cos(yu)*jnp.exp(-2*nu*t)
v_exact=-jnp.cos(xv)*jnp.sin(yv)*jnp.exp(-2*nu*t)
rmse=jnp.sqrt(jnp.mean((u_final-u_exact)**2+(v_final-v_exact)**2))
print('Velocity RMSE:',float(rmse),' max divergence:',float(jnp.max(jnp.abs(div(u_final,v_final,dx,dy)))))
assert float(rmse)<2e-2
assert float(jnp.max(jnp.abs(div(u_final,v_final,dx,dy))))<1e-10

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(11,3.3),constrained_layout=True)
for ax,field,title in zip(axs,[u_exact,u_final,u_final-u_exact],['u exact','u numerical','u error']):
 im=ax.imshow(np.asarray(field).T,origin='lower',extent=(0,L,0,L),cmap='RdBu_r'); ax.set_title(title);fig.colorbar(im,ax=ax,shrink=.75)
plt.show()

## 4. 数值验证与作业

- 令 $\nu=0$，观察能量守恒情况。为什么中心差分 + 显式 RK2 在复杂高 Re 流动下仍可能需要更稳定的通量和更小时间步？
- 将网格依次改为 24、48、96（并相应减小时间步），比较解析解误差。
- 修改 `advection()` 插值，尝试直观解释四点插值的几何位置。

**本课通过条件**：解析解 RMSE 足够小，且压力投影后的离散散度接近机器精度。